**Exercise 5.1.** Recall that S4 encodes the orientation-preserving isometries of the cube; there is one such
rotation for each permutation of the 4 diagonals of the cube. In particular, S4 acts (a) on the 8 vertices of
the cube and (b) on the 12 edges of the cube.
a) Decompose the permutation representation of S4 on the vertices of the cube into irreducibles.
b) Decompose the permutation representation of S4 on the edges of the cube into irreducibles.


**Exercise 5.2.** Compute the character table of 
$\mathcal{D}_{10}$.


**Exercise 5.3.** In this problem we compute the character table of 
$\mathcal{S}^5$. We start with only knowledge of the conjugacy classes and their sizes:


a) Let $U$, $U'$, and $V$ be the trivial, alternating, and standard representations of 
$\mathcal{S}_5$, respectively. Find the characters of these representations and confirm that they are irreducible.

b) Set 
$V'=V\otimes U'$, find its character, and confirm that it is a distinct irreducible representation from
$U$, $U'$, and $V$.

c) If $Y$ is a representation of $G$, then we can compute the characters of its symmetric powers 
$\mathrm{Sym}^k(Y)$ (c.f. Exercise 4.3) in terms of $\chi_Y$ . In particular, you may use the following formula without proof:
$$
\chi_{\mathrm{Sym}^2(Y)}(g) = \frac{1}{2}(\chi_Y(g)^2 + \chi_Y(g^2)).
$$
Compute the character for 
$\mathrm{Sym}^2(V)$ (with $V$ the standard representation) and use this to show that 
$\mathrm{Sym}^2(V)$ is a direct sum of three distinct irreducible representations.

d) Show that 
$\mathrm{Sym}^2(V)$ is the direct sum of $U$, $V$, and one previously unknown irreducible—call it $W$.

e) Complete the character table for 
$\mathcal{S}^5$



**Exercise 5.4.** Consider the class function 
$f: \mathcal{S}^5 \to \mathbb{C}$, where $f(\sigma)$ counts the number of 2-cycles in the
cycle notation for $\sigma$, e.g.:
$$
f((2 3 4)) = 0,\\
f((1 5)) = 1, \\
f((1 2)(3 4)) = 2.
$$
Write $f$ in the basis of irreducible characters for 
$\mathcal{C} (\mathcal{S}^5)$.


**Exercise 5.5.** If $V$ and $W$ are irreducible representations of $G$ and dim W = 1, show that $V \otimes W$ is also
an irreducible representation of $G$.

**Exercise 5.6.** (Bonus) If 
$\rho: G \to \mathrm{GL}(V )$ is faithful, show that every irreducible representation of $G$ is contained in some tensor power 
$V^{\otimes k}$.

Hint: If $W$ is an irreducible representation, consider the generating function
$$
\sum_{k=0}^\infty\langle\chi_W, \chi_{V^{\otimes k}}\rangle t^k
$$
If you can show this series converges to a nontrivial rational function, it must contain (infinitely many) non-zero coefficients!



In [2]:
import numpy as onp
import itertools as it
from collections import defaultdict

from pathlib import Path
import sys
_project_root = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
import src.symmetry as sym
from src.bases import CanonBasis3

B = CanonBasis3()

In [3]:
# conversion to int utils
def int_vec(v):
  return tuple(onp.rint(onp.asarray(v)).astype(int).tolist())

def int_mat(M):
  return onp.rint(onp.asarray(M)).astype(int)


### Exercise 5.1 helper

This block uses `src.symmetry` in the most direct way available here:

- generate the 24 orientation-preserving cube rotations as a concrete subgroup of `SO(3)`;
- let those rotations act on the 8 vertices, 12 edges, and 4 body diagonals of the cube;
- use the diagonal action to identify each rotation with a conjugacy class of `S_4`;
- compute the permutation characters on vertices and edges by taking traces;
- decompose those characters against the known irreducible character table of `S_4`.

So the library is helping with the group generation and concrete matrix actions, while the actual irreducible decomposition is done by character inner products.

In [ ]:
from typing import Any


def cycle_type(p):
  r""" cycle type from permutation `p` """
  seen = set()
  parts = []
  for i in range(len(p)):
    if i in seen:
      continue
    j = i
    k = 0
    while j not in seen:
      seen.add(j)
      j = p[j]
      k += 1
    parts.append(k)
  return tuple(sorted(parts, reverse=True))


def perm_hom_tup(items, image_fn):
  r""" permutation represented as tuple """
  lookup = {item: i for i, item in enumerate(items)}
  return tuple(lookup[image_fn(item)] for item in items)

def perm_hom(perm):
  r""" permutation representation homom """
  n = len(perm)
  M = onp.zeros((n, n), dtype=int)
  for j, i in enumerate(perm):
    # TODO why flipped?
    M[i, j] = 1
  return M


def compose_perm(p, q) -> tuple[Any, ...]:
  return tuple(p[q[i]] for i in range(len(p)))

def perm_power(p, k):
  out = tuple(range(len(p)))
  for _ in range(k):
    out = compose_perm(p, out)
  return out


def sign_perm(p):
  inversions = 0
  for i in range(len(p)):
    for j in range(i + 1, len(p)):
      inversions += p[i] > p[j]
  return -1 if inversions % 2 else 1


def perm_char(p):
  return int(onp.trace(perm_hom(p)))


In [ ]:

Vert3D = tuple[int, int, int]
Edge3D = tuple[Vert3D, Vert3D]

def init_cube() -> tuple[list[tuple[int, ...]], list[Edge3D]]:
  # TODO what is this?
  something: tuple[int, int] = (-int(1), int(1))
  verts = sorted(it.product(something, repeat=3))
  edges = []
  for u, v in it.combinations(verts, 2):
    if sum(a != b for a, b in zip(u, v)) == 1:
      edges.append(tuple(sorted((u, v))))
  edges = sorted(edges)
  return verts, edges


# Four body diagonals, represented by one endpoint direction for each line.
body_diagonals = [
  (1, 1, 1),
  (1, 1, -1),
  (1, -1, 1),
  (-1, 1, 1),
]



def diag_index(v):
  v = int_vec(v)
  for i, d in enumerate(body_diagonals):
    if v == d or v == tuple(-x for x in d):
      return i
  raise ValueError(f"vector {v} did not land on a body diagonal")



**Example 8.4** (Character table of 𝒮4).
 Derivation
First we use Proposition 8.2 to determine the 1-dimensional representations, using that
𝒮4
Ab = 𝒮4/𝒜4 ≅ 𝒞2.
The cyclic group of order 2 has two irreducible representations, 𝑈0,1 and 𝑈1,1, which lift to the trivial
𝑈 and alternating 𝑈′ representations, respectively. Next, we consider the standard representation
𝑉 . Recalling that the permutation representation 𝒮4 ⟳ ℂ4 decomposes as 𝑈 ⊕ 𝑉 , we have
𝜒𝑉 = 𝜒ℂ4 − 1; thanks to Exercise 0.6, we know that
𝜒ℂ4 (𝜎) = |Fix(𝜎)| .
Since the conjugacy classes of 𝒮4 are given by cycle types (Example 3.20), we have the partially
completed character table:
size class 1 6 8 6 3
𝕀 (1 2) (1 2 3) (1 2 3 4) (1 2)(3 4)
𝑈 1 1 1 1 1
𝑈′ 1 −1 1 −1 1
𝑉 3 1 0 −1 −1

Where we confirm that 𝑉 is irreducible using
⟨𝜒𝑉 , 𝜒𝑉 ⟩ = 1
24 (1 ⋅ 32 + 6 ⋅ 12 + 8 ⋅ 02 + 6 ⋅ (−1)2 + 3 ⋅ (−1)2) = 1.
Now we appeal to a general trick: that the tensor product of a 1-dimensional representation with
an irreducible representation must also be irreducible (see Exercise 5.5). In particular, setting
𝑉 ′ = 𝑉 ⊗ 𝑈′ yields a new representation whose character is given by 𝜎 ↦ 𝜒𝑉 (𝜎) sgn(𝜎). Since
|𝒮4| − (12 + 12 + 32 + 32) = 4,
and we have exhausted all 1-dimensional representations, we know that a 2-dimensional represen-
tation 𝑊 remains to be identified. We can compute its character without knowing anything about
what 𝑊 actually looks like. Indeed, we know that 𝜒𝑊 is orthogonal to the other four irreducible
characters, i.e.,
Span{𝜒𝑊 } = Span{𝜒𝑈 , 𝜒𝑈′ , 𝜒𝑉 , 𝜒𝑉 ′ }⟂
,
under the inner product Equation 7.4, since
dim C (𝒮4) = 5 = # number of conjugacy classes of 𝒮4.
We can compute this orthogonal complement using Equation 8.1: writing 𝑀 for the partially
completed character table, we want to find a vector orthogonal to its rows as weighted by the
conjugacy classes.
Listing 8.1 Computing the missing row of the character table
partial_table = Matrix([
[1, 1, 1, 1, 1],
[1,
-1, 1,
-1, 1],
[3, 1, 0,
-1,
-1],
[3,
-1, 0, 1,
-1],
])
weights = diagonal_matrix([1, 6, 8, 6, 3]) / 24
(partial_table.conjugate() * weights).right_kernel().basis()
[(1, 0, -1/2, 0, 1)]
This result is well-defined up to scaling, so we multiply by 2 to complete the character table.

In [ ]:
# A 90-degree face rotation and a 120-degree body-diagonal rotation generate
# the full orientation-preserving cube rotation group of order 24.
axisz = B.e3
axis111 = onp.array(sum(B))
cube_group = sym.generated_group(
  [
      sym.rotation3d(onp.pi / 2, axisz, label="r_z_90"),
      sym.rotation3d(2 * onp.pi / 3, axis111, label="r_111_120"),
  ],
  name="cube_rotations",
  max_order=24,
)
print("cube rotation group order =", cube_group.order)

verts, edges = init_cube()


# Order classes as in the standard S4 character table.
# import dataclasses as dcls
# @dcls.dataclass
# class S4Class:


I = (1, 1, 1, 1)
# two_swap = ((1,2), 1, 1)
two_swap = (2, 1, 1)
# three_cycle = ((1,2,3), 1)
three_cycle = (3, 1)
# four_cycle = (1,2,3,4)
four_cycle = (4,)
# swap_disjoint = ((1,2), (3,4))
swap_disjoint = (2, 2)

class_sizes_S4 = {
  I: 1, 
  two_swap: 6, 
  swap_disjoint: 3,
  three_cycle: 8, 
  four_cycle: 6
}

class_order_S4 = class_sizes_S4.keys()

# TODO should contain the order, sizes and stuff 
# and maybe create the character classes?
class GroupRep:
  def __init__(self):
    pass

  @property
  def order(self): ...

class CharS4:
  def __init__(self, data: dict):
    self.data = data

  def __getitem__(self, key):
    return self.data[key]

  def __or__(self, other):
    r""" Inner product on S4!!! """
    group_order = sum(class_sizes_S4.values())
    return sum(class_sizes_S4[ct] * self[ct] * other[ct] for ct in class_sizes_S4) / group_order
  
  def values_on_classes(self):
    return [self[ct] for ct in class_order_S4]

  def decompose(self, irrep_table):
    return {
      name: (self|irrep)
      for name, irrep in irrep_table.items()
    }

  @classmethod
  def from_row(cls, row: list[int]):
    return cls(dict(zip(class_order_S4, row)))


vert_char = {}
edge_char = {}
class_counts = defaultdict(int)

for op in cube_group.operations:
  M = int_mat(op.matrix)

  vertex_perm = perm_hom_tup(
    verts, 
    lambda v: int_vec(M @ onp.array(v))
  )
  edge_perm = perm_hom_tup(
      edges,
      lambda e: tuple(sorted(
        (int_vec(M @ onp.array(e[0])), int_vec(M @ onp.array(e[1])))
      )),
  )
  diag_perm = tuple(
    diag_index(M @ onp.array(d)) for d in body_diagonals
  )

  class_type = cycle_type(diag_perm)
  class_counts[class_type] += 1
  vert_char.setdefault(class_type, perm_char(vertex_perm))
  edge_char.setdefault(class_type, perm_char(edge_perm))

vert_char = CharS4(vert_char)
edge_char = CharS4(edge_char)

print("class sizes recovered from the diagonal action:")
for ct in class_order_S4:
  print(ct, class_counts[ct])

print("\nvertex permutation character on S4 classes:")
print([vert_char[ct] for ct in class_order_S4])

print("\nedge permutation character on S4 classes:")
print([edge_char[ct] for ct in class_order_S4])

# Hard-coded irreducible character table of S4 in the same class order.
table = {
  "triv":      [1,  1,  1,  1,  1],
  "sign":      [1, -1,  1,  1, -1],
  "two_dim":   [2,  0,  2, -1,  0],
  "std":       [3,  1, -1,  0, -1],
  "std_sign":  [3, -1, -1,  0,  1],
}

# irreducible characters for S_4
irred_chars_S4 = {
  char: CharS4.from_row(row) for char, row in table.items()
}

print("\nvertex decomposition:")
print(vert_char.decompose(irred_chars_S4))

print("\nedge decomposition:")
print(edge_char.decompose(irred_chars_S4))


cube rotation group order = 24
class sizes recovered from the diagonal action:
(1, 1, 1, 1) 1
(2, 1, 1) 6
(2, 2) 3
(3, 1) 8
(4,) 6

vertex permutation character on S4 classes:
[8, 0, 0, 2, 0]

edge permutation character on S4 classes:
[12, 2, 0, 0, 0]

vertex decomposition:
{'triv': 1.0, 'sign': 1.0, 'two_dim': 0.0, 'std': 1.0, 'std_sign': 1.0}

edge decomposition:
{'triv': 1.0, 'sign': 0.0, 'two_dim': 1.0, 'std': 2.0, 'std_sign': 1.0}


### Exercise 5.3 / 5.4 helper

This block is more plain permutation combinatorics than `src.symmetry`, but it still follows the same representation-theory workflow:

- generate `S_5` concretely as permutations of `{0,1,2,3,4}`;
- group elements by cycle type to recover the conjugacy classes and their sizes;
- compute the characters of `U`, `U'`, `V`, `V'`, and `Sym^2(V)` class-by-class;
- define `W = Sym^2(V) - U - V` and check its norm/orthogonality;
- define the class function from Exercise 5.4 and provide a generic decomposition helper once you have a full irreducible character basis.

So this should be useful for `5.3(a-d)` immediately, and for `5.4` once you finish the full `S_5` character table in `5.3(e)`.

In [ ]:
S5 = list(it.permutations(range(5)))

def fixed_points(p) -> int:
  return sum(i == p[i] for i in range(len(p)))

# TODO add classes to my group structure
classes_S5 = defaultdict(list)
for p in S5:
  classes_S5[cycle_type(p)].append(p)

IS5 = (1, 1, 1, 1, 1)
class_order_S5 = [
  IS5,
  (2, 1, 1, 1),
  (2, 2, 1),
  (3, 1, 1),
  (3, 2),
  (4, 1),
  (5,),
]
class_sizes_S5 = {ct: len(classes_S5[ct]) for ct in class_order_S5}

print("S5 conjugacy classes and sizes:")
for ct in class_order_S5:
  print(ct, class_sizes_S5[ct])

# class L2_S5:
#   def __init__(self, class_order_S5, class_sizes_S5):
#     self.class_order = class_order_S5
#     self.class_sizes = class_sizes_S5
  
#   def __or__(self, other):

class CharS5:
  def __init__(self, data: dict):
    self.data = data

  def __getitem__(self, key):
    return self.data[key]

  def __or__(self, other):
    r""" Inner product on S5!!! """
    return sum(class_sizes_S5[ct] * self[ct] * other[ct] for ct in class_order_S5) / len(S5)
  
  def values_on_classes(self):
    return [self[ct] for ct in class_order_S5]

  def decompose(self, irreducible_basis: dict[str, 'CharS5']):
    r""" decompose_class_function """
    return {name: (self|chi) for name, chi in irreducible_basis.items()}


chi_U = CharS5({ct: 1 for ct in class_order_S5})
chi_U_alt = CharS5({ct: sign_perm(classes_S5[ct][0]) for ct in class_order_S5})
chi_perm5 = CharS5({ct: fixed_points(classes_S5[ct][0]) for ct in class_order_S5})
chi_V = CharS5({ct: chi_perm5[ct] - 1 for ct in class_order_S5})
chi_V_alt = CharS5({ct: chi_V[ct] * chi_U_alt[ct] for ct in class_order_S5})
chi_sym2V = CharS5({
    ct: (chi_V[ct] ** 2 + chi_V[cycle_type(perm_power(classes_S5[ct][0], 2))]) / 2
    for ct in class_order_S5
})
chi_W = CharS5({ct: chi_sym2V[ct] - chi_U[ct] - chi_V[ct] for ct in class_order_S5})


print("\nCharacter values on S5 classes:")
for name, chi in [
  ("U", chi_U),
  ("U'", chi_U_alt),
  ("V", chi_V),
  ("V'", chi_V_alt),
  ("Sym^2(V)", chi_sym2V),
  ("W = Sym^2(V) - U - V", chi_W),
]:
  print(name, chi.values_on_classes(), "norm=", (chi|chi))

known_irreps_S5 = {
  "U": chi_U, "U'": chi_U_alt, "V": chi_V, "V'": chi_V_alt, "W": chi_W,
}

print("\nPairings with Sym^2(V):")
for name, chi in known_irreps_S5.items():
  print(name, (chi_sym2V|chi))

# Exercise 5.4: class function counting the number of 2-cycles.
f_two_cycle_count = CharS5({ct: sum(part == 2 for part in ct) for ct in class_order_S5})
print("\nValues of f on the S5 conjugacy classes:")
print(f_two_cycle_count.values_on_classes())

print("\nKnown projections of f onto the characters already identified in 5.3:")
print(f_two_cycle_count.decompose(known_irreps_S5))

# Once you finish 5.3(e), replace known_irreps_S5 by the full irreducible
# character table and the same helper will give the full answer to 5.4.


S5 conjugacy classes and sizes:
(1, 1, 1, 1, 1) 1
(2, 1, 1, 1) 10
(2, 2, 1) 15
(3, 1, 1) 20
(3, 2) 20
(4, 1) 30
(5,) 24

Character values on S5 classes:
U [1, 1, 1, 1, 1, 1, 1] norm= 1.0
U' [1, -1, 1, 1, -1, -1, 1] norm= 1.0
V [4, 2, 0, 1, -1, 0, -1] norm= 1.0
V' [4, -2, 0, 1, 1, 0, -1] norm= 1.0
Sym^2(V) [10.0, 4.0, 2.0, 1.0, 1.0, 0.0, 0.0] norm= 3.0
W = Sym^2(V) - U - V [5.0, 1.0, 1.0, -1.0, 1.0, -1.0, 0.0] norm= 1.0

Pairings with Sym^2(V):
U 1.0
U' 0.0
V 1.0
V' 0.0
W 1.0

Values of f on the S5 conjugacy classes:
[0, 1, 2, 0, 1, 0, 0]

Known projections of f onto the characters already identified in 5.3:
{'U': 0.5, "U'": 0.0, 'V': 0.0, "V'": 0.0, 'W': 0.5}


### Where the library is less direct

- `Exercise 5.2` is still mostly a manual character-table computation.
- `Exercise 5.5` and `Exercise 5.6` are proof problems; code can support examples, but it will not replace the argument.

So the most useful computational targets in this notebook are `5.1`, `5.3(c,d)`, and then `5.4` once the `S_5` table is complete.